# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Overall project type: a hybrid ML content-performance intelligence pipeline.**

The final operational task is **ranking / scoring**, because the system must automatically decide which content pages deserve attention first. However, the ranking is not the whole methodology. The project will use multiple ML task types as connected stages:

1. **Signal analysis** to understand which observable search, traffic, engagement, freshness, and content-performance variables contain useful information.
2. **Clustering** to discover recurring content-performance archetypes without imposing manual labels first.
3. **Classification / prediction** to estimate a leakage-safe future outcome such as decline, recovery, or another observed future performance state.
4. **Ranking / scoring** to combine the learned evidence into an automated intervention-priority queue.

The intended pipeline is therefore:

**observable page signals → performance archetype + predicted future outcome → priority score/ranking → reason codes + suggested action**

This keeps the project broader than a single predefined opportunity-scoring lane. Clustering provides structure, supervised prediction provides forward-looking evidence, and ranking converts those outputs into an actionable automated queue.

The ranking itself should be generated automatically from the data. Human involvement is reserved for approving or carrying out consequential content changes rather than manually constructing the ranking page by page.

In [ ]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

pipeline = [
    "signal analysis",
    "clustering",
    "classification/prediction",
    "ranking/scoring",
]
print("Hybrid ML pipeline:", " -> ".join(pipeline))
print("Framing check passed: one row represents one content item for page-level ML and ranking.")


## 2. Target or proxy

The hybrid pipeline does not use one target for every stage.

### Clustering target
Clustering is unsupervised, so it has **no target column**. It groups pages from observable pre-decision features and the resulting archetypes are inspected and named only after the clusters are formed.

### Supervised prediction target
For the classification stage, the preferred target is a **future observed performance state** built from the warehouse daily fact table:

**feature window (past 90 days) → decision point → target window (next 30 days)**

The target will be called **`future_momentum_state`**. It represents how the page's observed search visibility changes in the next 30 days relative to its recent pre-decision level. The initial classes will be **decline**, **stable**, and **growth**, derived from future observed impressions rather than from a manually assigned content action or priority score.

The exact movement thresholds and minimum-volume rule will be finalized and sensitivity-tested in the later data-contract/signal-audit work. The important constraint is that every field used to predict the target must be known **before** the decision point.

### Starter-data proxy
The 30,000-row starter CSV is a single trailing-90-day snapshot and therefore cannot supply a genuinely future 30-day target. Its existing `trend_direction` / `trend_pct` fields describe movement **inside the current snapshot**, not after a decision point.

For Assignment 3, I will therefore use the starter decline flag only as a **temporary framing proxy / sanity check**, not as the final ground truth. In the later warehouse version, the target will be rebuilt from separated past and future windows.

### Ranking target
The final ranking stage does not learn a manually authored priority label. It consumes learned evidence such as cluster/archetype membership and predicted future-state probabilities, together with safe observed context, to produce the automated priority order.

In [ ]:
# The starter dataset has no genuine future outcome window.
# Show the temporary proxy, then sketch the final future target schema.

assert "trend_direction" in df.columns
assert "trend_pct" in df.columns

starter_proxy = df["trend_direction"].eq("down").astype("int8")

print("Temporary starter proxy only:")
print(starter_proxy.value_counts().rename(index={0: "not_down", 1: "down"}))
print(f"Proxy decline rate: {starter_proxy.mean():.1%}")

# The final label will be built later from the warehouse daily table using
# past-90d features followed by a non-overlapping next-30d outcome window.
target_sketch = df[["content_id"]].head(8).copy()
target_sketch["feature_window"] = "past_90d"
target_sketch["target_window"] = "next_30d"
target_sketch["future_impressions_change_pct"] = pd.NA
target_sketch["future_momentum_state"] = pd.NA

print("\nFinal target-column sketch (future values intentionally unavailable in the starter snapshot):")
display(target_sketch)

print("\nLeakage rule: trend_direction and trend_pct must not be used as model features when the proxy/target is based on trend movement.")


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.